In [0]:
dbutils.widgets.text("region_id","","select region")

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col
cust_schema= StructType([StructField("COUNTRY_ID", IntegerType(), True), StructField("NAME", StringType(), True), StructField("NATIONALITY", StringType(), True), StructField("COUNTRY_CODE", StringType(), True), StructField("ISO_ALPHA2", StringType(), True), StructField("CAPITAL", StringType(), True), StructField("POPULATION", StringType(), True), StructField("AREA_KM2", StringType(), True), StructField("REGION_ID", IntegerType(), True), StructField("SUB_REGION_ID", IntegerType(), True)])
df=spark.read.format("csv").schema(cust_schema).option("header","true").load("/Volumes/databricks_practice/inputdb/country_population")
#df.display()
region_list = [(row["REGION_ID"]) for row in df.select("REGION_ID").distinct().collect() if row["REGION_ID"] is not None]
print(region_list)



df.write.format("delta").mode("overwrite").partitionBy("REGION_ID").saveAsTable("databricks_practice.inputdb.tbl_country_population")

In [0]:
dbutils.widgets.text("region_id", "", "select region_id")
_region_id= dbutils.widgets.get("region_id")
region_df=spark.sql(f"""select * from databricks_practice.inputdb.tbl_country_population where REGION_ID = int({_region_id})""")
region_df.show(5)

In [0]:

_count=region_df.count()
if _count >0:
    for i in _region_id:
        region_df.write.format("delta").partitionBy("SUB_REGION_ID").mode("overwrite").saveAsTable(f"databricks_practice.outputdb.tbl_region_{i}_population")

In [0]:

dbutils.notebook.exit(_count)